# 02 — PCA Embeddings (Full Scale)

Fits PCA 15D on the full F and P existence matrices to produce
the species-level geographic distribution embeddings Vf and Vp.

**Inputs:**
- `stage4_F_existence_phenofield.csv` (6,466 species × 3,162 bins)
- `stage4_P_existence_corrected.csv` (24,939 species × 3,162 bins)

**Outputs:**
- `stage4_Vf_phenofield.csv` — plant embedding, 15D (39.9% variance explained)
- `stage4_Vp_corrected.csv` — pollinator embedding, 15D (46.2% variance explained)

**Note on variance explained:**
Vf (39.9%) is lower than Vp (46.2%) despite the plant side having more
structured phenological data. This reflects that PhenoField plant observations
are more geographically dispersed across CONUS, while GBIF pollinator records
are more concentrated around human population centers — a tighter, more
compressible spatial pattern. Both embeddings partially encode human geography
rather than pure ecological distribution structure.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from pathlib import Path
import gc

BASE    = Path("/scratch/ariana.l")
OLD_S4  = BASE / "Stage 4 Link Prediction Model"
NEW_S4  = BASE / "New Stage 4 Link Prediction Model"

F_PATH  = OLD_S4 / "stage4_F_existence_phenofield.csv"
P_PATH  = NEW_S4 / "stage4_P_existence_corrected.csv"
VF_OUT  = OLD_S4 / "stage4_Vf_phenofield.csv"
VP_OUT  = NEW_S4 / "stage4_Vp_corrected.csv"

N_COMPONENTS = 15

print("Paths OK")

In [ ]:
# PCA on F → Vf (plant embedding)
print("Loading F matrix...")
F_df = pd.read_csv(F_PATH, index_col=0)
print(f"  F shape: {F_df.shape}")

print("Fitting PCA on F...")
pca_f = PCA(n_components=N_COMPONENTS, svd_solver='randomized', random_state=42)
Vf_arr = pca_f.fit_transform(F_df.values)

explained_f = pca_f.explained_variance_ratio_.sum()
print(f"  Variance explained: {explained_f:.4f}")
# Expected: ~0.399

Vf_df = pd.DataFrame(Vf_arr, index=F_df.index,
                      columns=[f"PC{i+1}" for i in range(N_COMPONENTS)])
Vf_df.to_csv(VF_OUT)
print(f"  Saved → {VF_OUT}")
print(f"  Vf shape: {Vf_df.shape}")

del F_df, Vf_arr
gc.collect()

In [ ]:
# PCA on P → Vp (pollinator embedding)
print("Loading P matrix...")
P_df = pd.read_csv(P_PATH, index_col=0)
print(f"  P shape: {P_df.shape}")

print("Fitting PCA on P...")
pca_p = PCA(n_components=N_COMPONENTS, svd_solver='randomized', random_state=42)
Vp_arr = pca_p.fit_transform(P_df.values)

explained_p = pca_p.explained_variance_ratio_.sum()
print(f"  Variance explained: {explained_p:.4f}")
# Expected: ~0.462
# Higher than Vf — GBIF observation density concentrates around urban areas,
# making the pollinator distribution matrix more compressible.

Vp_df = pd.DataFrame(Vp_arr, index=P_df.index,
                      columns=[f"PC{i+1}" for i in range(N_COMPONENTS)])
Vp_df.to_csv(VP_OUT)
print(f"  Saved → {VP_OUT}")
print(f"  Vp shape: {Vp_df.shape}")

del P_df, Vp_arr
gc.collect()

In [ ]:
# Summary
print(f"Vf: {Vf_df.shape} | variance explained: {explained_f:.4f}")
print(f"Vp: {Vp_df.shape} | variance explained: {explained_p:.4f}")
print()
print("Both embeddings are ready for use in the model feature vectors.")
print("Proceed to 03_ppe_integration for f_curves and V_delta construction.")